# Pre-training prior samples

This notebook draws random (expression, dataset) pairs from the synthetic pre-training prior and
plots them. Inputs come from the TabICLv2 prior, constants are sampled bottom-up conditioned on
the data, targets carry heteroscedastic and structured noise, and inputs and targets are
standardized. The shown expressions are those the model is trained to predict, in terms of the
standardized data.

Setup: install the package with the `notebook` extra and the `data-gen` group as described in the
README's [Setup](../README.md#setup) section.

In [ ]:
import re

import matplotlib.pyplot as plt

from src.rasr.data.generation.data_creator import _remap_expression_variables, load_toml_config
from src.rasr.data.generation.dataset_sampler import DatasetSampler
from src.rasr.data.generation.expression_sampler import ExpressionSampler
from src.rasr.data.generation.expression_wrapper import ExpressionWrapper

config = load_toml_config("../src/rasr/train_configs/data_pre-train.toml")
expression_sampler = ExpressionSampler.from_config(config)
dataset_sampler = DatasetSampler.from_config(config)
expression_wrapper = ExpressionWrapper(config["prior"]["expressions"]["operator_arity"])
engine = expression_sampler.simplipy_engine


def to_infix(expression, decimals=2):
    infix = engine.prefix_to_infix(list(expression))
    infix = re.sub(r"(?<![a-zA-Z_])\d+\.\d+", lambda m: f"{float(m.group()):.{decimals}f}", infix)
    return infix.replace("+ -", "- ")


def sample_one(num_variables, num_samples=512, max_attempts=10):
    last_error = None
    for _ in range(max_attempts):
        try:
            skeleton, variables = expression_sampler.sample_skeleton_only_with_retries(
                retries=10, num_unique_symbols=num_variables
            )
            x, y, variable_mapping, _, constants, mu_y, sigma_y = dataset_sampler.sample_with_retries(
                skeleton=skeleton,
                variables=variables,
                num_samples=num_samples,
                retries=3,
                num_extra_columns=0,
            )
            expression, _ = expression_sampler.build_expression_from_skeleton(skeleton, constants, constants, variables)
        except Exception as error:
            last_error = error
            continue
        if x.shape[1] != num_variables:
            continue
        wrapped = expression_wrapper.wrap(expression, mu_y, sigma_y)
        return x, y, to_infix(_remap_expression_variables(wrapped, variable_mapping))
    raise RuntimeError("No valid sample found.") from last_error

## One variable

Each panel shows one dataset, titled with the expression it was generated from.

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(16, 12), constrained_layout=True)
for ax in axes.flat:
    x, y, infix = sample_one(num_variables=1)
    ax.scatter(x[:, 0], y, s=4, alpha=0.5, rasterized=True)
    ax.set_title(infix if len(infix) <= 45 else infix[:44] + "…", fontsize=8, family="monospace")
    ax.tick_params(labelsize=7)
    ax.grid(alpha=0.2)
plt.show()

## Two variables

The inputs of each dataset, colored by the target. The input distributions come from the TabICLv2
prior and are often skewed, clustered, or correlated.

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 12), constrained_layout=True)
for ax in axes.flat:
    x, y, infix = sample_one(num_variables=2)
    ax.scatter(x[:, 0], x[:, 1], c=y, s=4, cmap="viridis", rasterized=True)
    ax.set_title(infix if len(infix) <= 45 else infix[:44] + "…", fontsize=8, family="monospace")
    ax.set_xlabel("x1", fontsize=8)
    ax.set_ylabel("x2", fontsize=8)
    ax.tick_params(labelsize=7)
    ax.grid(alpha=0.2)
plt.show()